In [ ]:
import geemap
import sys
import ee
import os

from datetime import datetime, timedelta

ee.Authenticate()
ee.Initialize()

sys.path.append( r'E:\acolites\20231023.0' )

In [ ]:
import acolite

from gee_acolite.utils.search import search_list, search
from gee_acolite.correction import ACOLITE
from gee_acolite import bathymetry

settings = r'C:\Users\sergi\Documents\repos\gee_acolite\settings.txt'
corrector = ACOLITE(acolite=acolite, settings=settings)

In [ ]:
roi = ee.Geometry({"type":"Polygon","coordinates":[[[-7.729568,37.042294],[-7.890587,37.042294],[-7.890587,36.937587],[-7.729568,36.937587],[-7.729568,37.042294]]]})
roi

In [ ]:
map = geemap.Map()
map.centerObject(roi)
map.addLayer(roi)
map

In [ ]:
start_date = ['2018-09-19', '2018-09-22', '2018-09-24', '2018-10-04', '2018-10-07', '2018-10-17', '2018-10-19', '2018-10-22', ][::1]
end_date = ['2018-09-20', '2018-09-23', '2018-09-25', '2018-10-05', '2018-10-08', '2018-10-18', '2018-10-20', '2018-10-23', ][::1]
crs = 'EPSG:32629'
tile = 'T29SPA'

sentinel2_l1 = search_list(roi, start_date, end_date, tile=tile)
print(sentinel2_l1.size().getInfo())
sentinel2_l2w, l2_settings = corrector.correct(sentinel2_l1)
image = bathymetry.multi_image(sentinel2_l2w).clip(roi)

In [ ]:
vis_params = {
    'min': 0.9,
    'max': 1.2,
    'palette': ['440154', '482878', '3e4a89', '31688e', '26838f', '1f9e89', '35b779', '6ece58', 'b5de2b', 'fde725'][::-1]
}
map.addLayer(image.select('pSDB_green'), vis_params, 'pSDB_green')
map

In [ ]:
insitu_bathymetry = ee.Image('projects/ee-sergiohercar1/assets/InSituBathymetry/2018_10m').rename(['depth'])

vis_params = {
    'min': 0,
    'max': 20,
    'palette': ['440154', '482878', '3e4a89', '31688e', '26838f', '1f9e89', '35b779', '6ece58', 'b5de2b', 'fde725'][::-1]
}
map.addLayer(insitu_bathymetry.select('depth'), vis_params, 'depth')
map

In [ ]:
# Test calibrate_sdb function with random sampling
from gee_acolite.bathymetry import calibrate_sdb, apply_calibration

# Calibrate using 20 random points up to 10m depth
calibration_result = calibrate_sdb(
    psdb_image=image.select('pSDB_green'),
    insitu_bathymetry=insitu_bathymetry,
    region=roi,
    max_depth=10,
    num_points=20,
    seed=42,
    scale=10
)

print(f"Calibration results using {calibration_result['num_points']} random points:")
print(f"  Slope: {calibration_result['slope']:.4f}")
print(f"  Intercept: {calibration_result['intercept']:.4f}")
print(f"  Correlation: {calibration_result['correlation']:.4f}")

# Apply calibration to generate SDB
sdb_calibrated = apply_calibration(
    psdb_image=image.select('pSDB_green'),
    slope=calibration_result['slope'],
    intercept=calibration_result['intercept'],
    output_name='SDB_green_calibrated'
)

print(f"\nRegression equation: SDB = {calibration_result['slope']:.4f} * pSDB + {calibration_result['intercept']:.4f}")

Calibration results using 1 random points:


TypeError: unsupported format string passed to NoneType.__format__

In [ ]:
# Usar la imagen ya definida (clipeada al ROI)
psdb_image = image.select('pSDB_green')

# Combinar ambas imágenes para la regresión
combined = psdb_image.addBands(insitu_bathymetry.rename('depth'))

# Filtrar píxeles donde depth <= 10m (y > 0 para excluir no-data)
mask_10m = insitu_bathymetry.lte(10).And(insitu_bathymetry.gt(0))
combined_filtered = combined.updateMask(mask_10m)

# Realizar regresión lineal usando reduceRegion con todos los píxeles válidos hasta 10m
regression = combined_filtered.reduceRegion(
    reducer=ee.Reducer.linearFit(),
    geometry=roi,
    scale=10,
    maxPixels=1e9
)

# Mostrar resultados de la regresión
regression_info = regression.getInfo()
print(f"SDB Green = {regression_info['scale']:.4f} * pSDB_green + {regression_info['offset']:.4f}")

# Calcular correlación de Pearson
correlation = combined_filtered.reduceRegion(
    reducer=ee.Reducer.pearsonsCorrelation(),
    geometry=roi,
    scale=10,
    maxPixels=1e9
)
print(f"Correlación de Pearson: {correlation.getInfo()['correlation']:.4f}")

# Calcular MAE usando los coeficientes de regresión
scale_coef = ee.Number(regression.get('scale'))
offset_coef = ee.Number(regression.get('offset'))
predicted_depth = psdb_image.multiply(scale_coef).add(offset_coef)
absolute_error = predicted_depth.subtract(insitu_bathymetry).abs()

mae = absolute_error.updateMask(mask_10m).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=10,
    maxPixels=1e9
)
print(f"MAE (Mean Absolute Error): {mae.getInfo()['pSDB_green']:.3f} m")

# Contar número de píxeles válidos
count = combined_filtered.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=roi,
    scale=10,
    maxPixels=1e9
)
print(f"\nNúmero de píxeles válidos: {count.getInfo()}")

# Añadir banda SDB_green a image
sdb_green = psdb_image.multiply(scale_coef).add(offset_coef).rename('SDB_green')
image = image.addBands(sdb_green)
print(f"\nBandas en image: {image.bandNames().getInfo()}")

In [ ]:
vis_params = {
    'min': 0,
    'max': 15,
    'palette': ['440154', '482878', '3e4a89', '31688e', '26838f', '1f9e89', '35b779', '6ece58', 'b5de2b', 'fde725'][::-1]
}
map.addLayer(image.select('SDB_green').clip(roi), vis_params, 'SDB_green')
map

In [ ]:
# Calcular banda de error (SDB_green - insitu)
error_band = image.select('SDB_green').subtract(insitu_bathymetry.select('depth')).rename('error')
image = image.addBands(error_band)

# Muestrear valores de error para el histograma
error_sample = error_band.sample(
    region=roi,
    scale=10,
    numPixels=5000,
    seed=42,
    geometries=False
)

# Convertir a DataFrame y crear histograma
import matplotlib.pyplot as plt

df = geemap.ee_to_df(error_sample)
error_values = df['error'].dropna()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(error_values, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Error = 0')
ax.axvline(x=error_values.mean(), color='green', linestyle='-', linewidth=2, label=f'Media = {error_values.mean():.2f} m')
ax.set_xlabel('Error (SDB - In-Situ) [m]')
ax.set_ylabel('Frecuencia')
ax.set_title('Histograma de Error SDB vs In-Situ')
ax.legend()
ax.grid(True, alpha=0.3)

print(f"Media del error: {error_values.mean():.3f} m")
print(f"Desviación estándar: {error_values.std():.3f} m")
print(f"Error mínimo: {error_values.min():.3f} m")
print(f"Error máximo: {error_values.max():.3f} m")

plt.tight_layout()
plt.show()

In [ ]:
# Aplicar Optical Deep Water Model
import sys

# Añadir path local al principio para que tenga prioridad sobre site-packages
local_path = r'c:\Users\sergi\Documents\repos\gee_acolite'
if local_path not in sys.path:
    sys.path.insert(0, local_path)

# Eliminar módulos cargados previamente para forzar importación desde local
for mod in list(sys.modules.keys()):
    if 'gee_acolite' in mod:
        del sys.modules[mod]

from gee_acolite.bathymetry import optical_deep_water_model

# Seleccionar bandas necesarias para ODW
blue = image.select('Rrs_B2')   # Banda azul
green = image.select('Rrs_B3')  # Banda verde
vnir = image.select('Rrs_B5')   # Banda NIR

# Aplicar filtro ODW a SDB_green
sdb_green_odw = optical_deep_water_model(
    model=image.select('SDB_green'),
    blue=blue,
    green=green,
    vnir=vnir
).rename('SDB_green_ODW')

# Añadir banda a image
image = image.addBands(sdb_green_odw)
print(f"Bandas en image: {image.bandNames().getInfo()}")

In [ ]:
# Comparar histogramas de error: SDB_green vs SDB_green_ODW
import matplotlib.pyplot as plt

# Calcular errores para ambos modelos
error_sdb = image.select('SDB_green').subtract(insitu_bathymetry.select('depth'))
error_odw = image.select('SDB_green_ODW').subtract(insitu_bathymetry.select('depth'))

# Muestrear valores
sample_sdb = error_sdb.sample(region=roi, scale=10)
sample_odw = error_odw.sample(region=roi, scale=10)

# Convertir a DataFrame
df_sdb = geemap.ee_to_df(sample_sdb)
df_odw = geemap.ee_to_df(sample_odw)

error_sdb_values = df_sdb['SDB_green'].dropna()
error_odw_values = df_odw['SDB_green_ODW'].dropna()

# Crear figura con 2 histogramas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma SDB_green (sin ODW)
axes[0].hist(error_sdb_values, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].axvline(x=error_sdb_values.mean(), color='green', linestyle='-', linewidth=2, 
                label=f'Media = {error_sdb_values.mean():.2f} m')
axes[0].set_xlabel('Error (SDB - In-Situ) [m]')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title(f'SDB_green (sin ODW)\nn={len(error_sdb_values)}, MAE={abs(error_sdb_values).mean():.2f}m')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histograma SDB_green_ODW (con ODW)
axes[1].hist(error_odw_values, bins=50, edgecolor='black', alpha=0.7, color='darkorange')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].axvline(x=error_odw_values.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Media = {error_odw_values.mean():.2f} m')
axes[1].set_xlabel('Error (SDB - In-Situ) [m]')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title(f'SDB_green_ODW (con ODW)\nn={len(error_odw_values)}, MAE={abs(error_odw_values).mean():.2f}m')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparación de Error: Sin vs Con Optical Deep Water Filter', fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n=== Estadísticas SDB_green (sin ODW) ===")
print(f"  Media: {error_sdb_values.mean():.3f} m | Std: {error_sdb_values.std():.3f} m | MAE: {abs(error_sdb_values).mean():.3f} m")
print(f"\n=== Estadísticas SDB_green_ODW (con ODW) ===")
print(f"  Media: {error_odw_values.mean():.3f} m | Std: {error_odw_values.std():.3f} m | MAE: {abs(error_odw_values).mean():.3f} m")

In [ ]:
vis_params = {
    'min': 0,
    'max': 15,
    'palette': ['440154', '482878', '3e4a89', '31688e', '26838f', '1f9e89', '35b779', '6ece58', 'b5de2b', 'fde725'][::-1]
}
map.addLayer(image.select('SDB_green_ODW').clip(roi), vis_params, 'SDB_green_ODW')
map